# ACE Cluster Curation -> TCAV Concepts

This notebook does:
1. Load one ACE run output (`cluster_scores.csv`, `concepts_auto`)
2. Rank/filter clusters by AUC + size
3. Visual inspect top clusters
4. Export selected clusters to final concept folders
5. Build/update `random_0` baseline concept

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CANDIDATES = [
    Path('/home/SpeakerRec/BioVoice'),
    Path.cwd().resolve(),
    Path.cwd().resolve().parents[1] if len(Path.cwd().resolve().parents) > 1 else Path.cwd().resolve(),
]
ROOT = None
for c in CANDIDATES:
    if (c / 'resnet_293' / 'ace' / 'ace_a11_poc' / 'output').exists():
        ROOT = c
        break
if ROOT is None:
    raise RuntimeError('Could not locate BioVoice root. Set ROOT manually.')

ACE_BASE = ROOT / 'resnet_293' / 'ace' / 'ace_a11_poc'
ACE_OUTPUT = ACE_BASE / 'output'
FINAL_CONCEPTS_BASE = ROOT / 'concept' / 'resnet_293_ace_a11_concepts'

print('ROOT =', ROOT)
print('ACE_OUTPUT =', ACE_OUTPUT)
print('FINAL_CONCEPTS_BASE =', FINAL_CONCEPTS_BASE)

In [ ]:
# Select run folder (latest by default)
run_dirs = sorted([p for p in ACE_OUTPUT.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
if not run_dirs:
    raise RuntimeError(f'No run dirs in {ACE_OUTPUT}')

RUN_DIR = run_dirs[0]
# To use a specific run, set manually, for example:
# RUN_DIR = ACE_OUTPUT / 'a11_ace_poc_run1'

print('RUN_DIR =', RUN_DIR)
print('Has cluster_scores.csv:', (RUN_DIR / 'cluster_scores.csv').exists())
print('Has concepts_auto:', (RUN_DIR / 'concepts_auto').exists())

In [ ]:
# Rank and auto-filter
score_csv = RUN_DIR / 'cluster_scores.csv'
df = pd.read_csv(score_csv)
df = df.sort_values(['score_auc_spoof_vs_bona', 'cluster_size'], ascending=[False, False]).reset_index(drop=True)
display(df.head(20))

MIN_AUC = 0.65
MIN_SIZE = 80
cand = df[(df['score_auc_spoof_vs_bona'] >= MIN_AUC) & (df['cluster_size'] >= MIN_SIZE)].copy()
print(f'Candidates with AUC>={MIN_AUC} and size>={MIN_SIZE}:', len(cand))
display(cand.head(30))

In [ ]:
# Visual inspect top clusters using _preview_stack.npy mean map
CONCEPTS_AUTO = RUN_DIR / 'concepts_auto'
top_ids = df['cluster_id'].astype(int).head(12).tolist()

n = len(top_ids)
cols = 3
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.8 * rows), squeeze=False)

for i, cid in enumerate(top_ids):
    ax = axes[i // cols][i % cols]
    cdir = CONCEPTS_AUTO / f'cluster_{cid:03d}'
    p = cdir / '_preview_stack.npy'
    if not p.exists():
        ax.set_title(f'cluster_{cid:03d} (no preview)')
        ax.axis('off')
        continue
    arr = np.load(p)  # (N,F,T), padded
    mean_map = arr.mean(axis=0)
    im = ax.imshow(mean_map, aspect='auto', origin='lower')
    row = df[df['cluster_id'] == cid].iloc[0]
    ax.set_title(f'cluster_{cid:03d} | AUC={row.score_auc_spoof_vs_bona:.3f} | n={int(row.cluster_size)}')
    ax.set_xlabel('Time')
    ax.set_ylabel('Mel')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for j in range(n, rows * cols):
    axes[j // cols][j % cols].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Manual selection + naming (edit this)
# Example: use top candidates initially, then refine after visual check.
SELECTED_CLUSTER_TO_NAME = {
    # 3: 'ace_l3_a11_concept_01',
    # 7: 'ace_l3_a11_concept_02',
}

if not SELECTED_CLUSTER_TO_NAME:
    print('No manual selection yet. Filling with first 8 candidates for convenience.')
    temp_ids = cand['cluster_id'].astype(int).head(8).tolist()
    SELECTED_CLUSTER_TO_NAME = {cid: f'ace_l3_a11_concept_{i+1:02d}' for i, cid in enumerate(temp_ids)}

print('Selected clusters:')
for cid, name in SELECTED_CLUSTER_TO_NAME.items():
    print(f'  cluster_{cid:03d} -> {name}')

In [ ]:
# Export selected clusters into final concept set
FINAL_CONCEPTS_DIR = FINAL_CONCEPTS_BASE / 'concepts'
FINAL_CONCEPTS_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for cid, cname in SELECTED_CLUSTER_TO_NAME.items():
    src = CONCEPTS_AUTO / f'cluster_{int(cid):03d}'
    if not src.exists():
        print('Missing cluster dir:', src)
        continue
    dst = FINAL_CONCEPTS_DIR / cname
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True, exist_ok=True)

    files = sorted([p for p in src.glob('*.npy') if not p.name.startswith('_')])
    for j, fp in enumerate(files):
        shutil.copy2(fp, dst / f'{j:06d}.npy')

    row = df[df['cluster_id'] == int(cid)].iloc[0].to_dict() if (df['cluster_id'] == int(cid)).any() else {}
    row.update({'cluster_id': int(cid), 'concept_name': cname, 'num_files': len(files)})
    rows.append(row)
    print(f'Exported {len(files)} samples: cluster_{int(cid):03d} -> {dst.name}')

sel_df = pd.DataFrame(rows)
sel_df.to_csv(FINAL_CONCEPTS_BASE / 'selected_concepts.csv', index=False)
print('Wrote:', FINAL_CONCEPTS_BASE / 'selected_concepts.csv')

In [ ]:
# Build or update random_0 baseline concept
# Strategy: sample random .npy from non-selected clusters.
RANDOM_COUNT = 200
selected_ids = set(int(k) for k in SELECTED_CLUSTER_TO_NAME.keys())
other_cluster_dirs = [
    p for p in sorted(CONCEPTS_AUTO.glob('cluster_*'))
    if p.is_dir() and int(p.name.split('_')[-1]) not in selected_ids
]

pool = []
for cdir in other_cluster_dirs:
    files = [p for p in cdir.glob('*.npy') if not p.name.startswith('_')]
    pool.extend(files)

if len(pool) < RANDOM_COUNT:
    print(f'Warning: pool has only {len(pool)} files; using all of them')
    picked = pool
else:
    random.seed(1337)
    picked = random.sample(pool, RANDOM_COUNT)

rand_dir = FINAL_CONCEPTS_DIR / 'random_0'
if rand_dir.exists():
    shutil.rmtree(rand_dir)
rand_dir.mkdir(parents=True, exist_ok=True)

for i, fp in enumerate(picked):
    shutil.copy2(fp, rand_dir / f'{i:06d}.npy')

print('random_0 size =', len(picked))
print('random_0 dir =', rand_dir)

In [ ]:
# Save export meta
meta = {
    'run_dir': str(RUN_DIR),
    'selected_clusters': {str(k): v for k, v in SELECTED_CLUSTER_TO_NAME.items()},
    'final_concepts_dir': str(FINAL_CONCEPTS_DIR),
    'random_0_dir': str(FINAL_CONCEPTS_DIR / 'random_0'),
}
with (FINAL_CONCEPTS_BASE / 'export_meta.json').open('w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)
print('Wrote:', FINAL_CONCEPTS_BASE / 'export_meta.json')